In [1]:
# Set up the file path
import os
os.chdir('..')

In [2]:
# Import packages
from RL4CRN.iocrns.reaction_library import construct_hill_production_library, construct_mass_action_library
from RL4CRN.policies.parameter_generator_from_distribution import ParameterGeneratorFromDistribution
from RL4CRN.utils.utils import batch_multi_hot
import torch
import numpy as np

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
species_labels = ['X_1', 'X_2', 'X_3']
n = len(species_labels)
P = 1
R = 2
O = 1
library = construct_hill_production_library(species_labels, max_product_order=P, max_num_regulators=R)
library_temp = construct_mass_action_library(species_labels, order=O)
print(library)
print("------------------------")
print(library_temp)
print("------------------------")
library.merge(library_temp)
print(library)

Number of reactions: 54
R0: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_1(None, 1))]
R1: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_2(None, 1))]
R2: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_3(None, 1))]
R3: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = X_1(None, 1);    (Kr, nr) = )]
R4: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = X_2(None, 1);    (Kr, nr) = )]
R5: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = X_3(None, 1);    (Kr, nr) = )]
R6: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_1(None, 1), X_2(None, 1))]
R7: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_1(None, 1), X_3(None, 1))]
R8: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_2(None, 1), X_3(None, 1))]
R9: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = X_1(None, 1);    (Kr, nr)

In [5]:
# Construct the continuous parameter mask
continuous_parameter_mask = library.get_parameter_mask(mode='continuous', force=False)
continuous_parameter_mask = torch.tensor(continuous_parameter_mask, dtype=torch.float32).to(device) if continuous_parameter_mask is not None else None # Shape: (M, max_num_continuous_parameters)
print("Continuous parameter mask:")
print(f"Type: {type(continuous_parameter_mask)}, Shape: {continuous_parameter_mask.shape}, dtype:, {continuous_parameter_mask.dtype}")
print(continuous_parameter_mask)

Continuous parameter mask:
Type: <class 'torch.Tensor'>, Shape: torch.Size([67, 4]), dtype:, torch.float32
tensor([[1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [

In [ ]:
# Create a continuous parameter generator with lognormal distribution
deep_layer_size = 10 * 1024
M = len(library)
parameter_head_attributes = {"hidden_size": 128, "num_layers": 5}
# distribution = {'type': 'lognormal', 'dim': continuous_parameter_mask.shape[1]}
distribution = {'type': 'lognormal_processed', 'dim': continuous_parameter_mask.shape[1], 
                 'mu_max': 5.0, 'sigma_min': 1e-5, 'sigma_max': 5.0, 'rho_max': 0.9}
backbone_attributes={  "input_size": deep_layer_size + M, 
                        "hidden_size": parameter_head_attributes["hidden_size"], 
                        "num_layers": parameter_head_attributes["num_layers"]
                        }
parameter_generator_continuous = ParameterGeneratorFromDistribution(distribution, backbone_attributes, device=device)

In [7]:
# Generate random reaction indices and mask subset
N = 5
samples_reaction_idx = torch.randint(0, M, (N,))
print(f"Sampled reaction indeces: {samples_reaction_idx}")
print("Sampled reactions:")
library.print_reactions(samples_reaction_idx) 
continuous_parameter_mask_subset = continuous_parameter_mask[samples_reaction_idx] if continuous_parameter_mask is not None else None # Shape: (N, max_num_continuous_parameters) or None
print("Continuous parameter mask subset:")
print(continuous_parameter_mask_subset)

Sampled reaction indeces: tensor([65, 62, 49, 45,  1])
Sampled reactions:
Number of reactions: 5
R65: X_3 ----> X_1;  [MAK(None)]
R62: X_2 ----> X_1;  [MAK(None)]
R49: ∅ ----> X_3;  [HILLProd(b = None, Vm = None,    (Ka, na) = X_3(None, 1);    (Kr, nr) = X_1(None, 1))]
R45: ∅ ----> X_3;  [HILLProd(b = None, Vm = None,    (Ka, na) = X_1(None, 1);    (Kr, nr) = X_2(None, 1))]
R1: ∅ ----> X_1;  [HILLProd(b = None, Vm = None,    (Ka, na) = ;    (Kr, nr) = X_2(None, 1))]
Continuous parameter mask subset:
tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 0.]], device='cuda:0')


In [8]:
# Generate parameters
samples_reaction_hot = batch_multi_hot(samples_reaction_idx.unsqueeze(-1).cpu().numpy(), len(library), intensities=None, device=device) # shape: (N, M)
encoded = 100 * torch.rand(N, deep_layer_size).to(device=device)  # Dummy encoded representation
x = torch.cat([encoded, samples_reaction_hot], dim=-1)
samples, log_probs, entropies = parameter_generator_continuous(x, continuous_parameter_mask_subset, samples=None)
print('samples:\n', samples)
print('log_probs:\n', log_probs)
print('entropies:\n', entropies)

samples:
 tensor([[2.5066e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.9530e-08, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.5977e+02, 4.2161e-14, 1.0050e+00, 0.0000e+00],
        [3.3612e-34,        inf,        inf, 7.9515e+18],
        [6.8375e+06, 9.6751e-07, 5.0192e-05, 0.0000e+00]], device='cuda:0')
log_probs:
 tensor([  -2.7370,   13.4317,  115.5013, -388.1711,   -4.7077],
       device='cuda:0', grad_fn=<IndexPutBackward0>)
entropies:
 tensor([ 1.5210,  3.1519, 10.9729, 11.4394,  1.1149], device='cuda:0',
       grad_fn=<IndexPutBackward0>)


In [43]:
d = 4
rho_max = 1
off_raw = torch.randint(-10, 10, (N, (d**2 - d)//2), dtype=torch.float32, device=device)
T = torch.zeros(N, d, d, device=device)
ar = torch.arange(d, device=device)
tril_i, tril_j = torch.tril_indices(d, d, offset=-1)
T[:, ar, ar] = 1.0
T[:, tril_i, tril_j] = rho_max * torch.tanh(off_raw)
print(off_raw)
print(T)

tensor([[ -8.,  -5.,  -8.,  -1.,   0.,   5.],
        [ -1.,   2., -10.,   1., -10., -10.],
        [ -9.,   4.,   0., -10.,  -6., -10.],
        [  7.,   1.,   8.,   4.,  -7.,   4.],
        [  9.,   7.,   4.,   8.,   2.,   8.]], device='cuda:0')
tensor([[[ 1.0000,  0.0000,  0.0000,  0.0000],
         [-1.0000,  1.0000,  0.0000,  0.0000],
         [-0.9999, -1.0000,  1.0000,  0.0000],
         [-0.7616,  0.0000,  0.9999,  1.0000]],

        [[ 1.0000,  0.0000,  0.0000,  0.0000],
         [-0.7616,  1.0000,  0.0000,  0.0000],
         [ 0.9640, -1.0000,  1.0000,  0.0000],
         [ 0.7616, -1.0000, -1.0000,  1.0000]],

        [[ 1.0000,  0.0000,  0.0000,  0.0000],
         [-1.0000,  1.0000,  0.0000,  0.0000],
         [ 0.9993,  0.0000,  1.0000,  0.0000],
         [-1.0000, -1.0000, -1.0000,  1.0000]],

        [[ 1.0000,  0.0000,  0.0000,  0.0000],
         [ 1.0000,  1.0000,  0.0000,  0.0000],
         [ 0.7616,  1.0000,  1.0000,  0.0000],
         [ 0.9993, -1.0000,  0.9993,  1.0